## Topics:
- Logistic Regression
- Log Loss
- Process to update Weights
    - numpy process
    - keras process
        - tf.keras.Sequential()
        - tf.keras.layers.Dense()
- history.history['loss'][0]
- model.compile()
- Tensorflow - functional API
#
---

### **Logistic Regression:**

Logistic regression is a statistical method for modeling the relationship between a set of input features and a binary (two-category) outcome. Despite its name, it's a **classification** algorithm, not a regression technique.

#### **Key Concepts**

**The Problem It Solves**

Logistic regression predicts the probability that an observation belongs to one of two classes (e.g., spam/not spam, fraud/legitimate, diseased/healthy). The output is a probability between 0 and 1, which you can threshold at 0.5 to make a binary classification.

**The Sigmoid Function**

At its core, logistic regression applies the **sigmoid function** to a linear combination of features:

```
P(y=1|x) = 1 / (1 + e^(-z))

where z = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ
```

This S-shaped curve transforms any input into a probability. When z=0, the probability is 0.5; as z increases, P approaches 1; as z decreases, P approaches 0.

**How It Learns**

Logistic regression finds coefficients (β₀, β₁, ..., βₙ) that maximize the **likelihood** of the observed data. This is typically done via maximum likelihood estimation (MLE) using optimization algorithms like gradient descent.

#### **Advantages**

- Simple, interpretable, and computationally efficient
- Probabilistic output (not just a hard classification)
- Works well for linearly separable data
- Coefficients tell you the direction/magnitude of feature influence
- Fast to train and predict

#### **Limitations**

- **Assumes a linear relationship between features and log-odds: z = log(p/(1-p)**
- Struggles with non-linear patterns (use neural networks or kernel methods instead)
- Not ideal for highly imbalanced datasets without adjustment
- Can overfit with high-dimensional data (use regularization: L1/L2)
#
---

Let's write out our model function:

\begin{align}
h_W(x) = \phi(w_0x_0 + w_1x_1 + w_2x_2 + w_3x_3) = \phi(xW^T) = \frac{1}{1+e^{(-xW^T)}}
\end{align}

We can get all predictions with this matrix product:

\begin{align}
\hat{Y} = h_W(X) = \phi(XW^T) =
\phi\begin{pmatrix}
x_{0,0} & x_{0,1} & x_{0,2} & x_{0,3} \\
x_{1,0} & x_{1,1} & x_{1,2} & x_{1,3} \\
\vdots & \vdots & \vdots & \vdots \\
x_{m-1,0} & x_{m-1,1} & x_{m-1,2} & x_{m-1,3} \\
\end{pmatrix}
\begin{pmatrix}
w_0 \\
w_1 \\
w_2 \\
w_3 \\
\end{pmatrix}
\end{align}

First let's write the sigmoid (logistic) function $\phi$.

We're not going to use MSE for logistic regression. Instead, we'll use the *logistic loss*, also called *binary cross-entropy* (more on that name later):

\begin{align}
LogLoss = \frac{1}{m} \sum_i -y_i\log(\hat{y_i}) - (1-y_i)\log(1-\hat{y_i})
\end{align}

Despite this new loss function, it turns out that the gradient computation is the same as it was for MSE with linear regression. A happy coincidence.

\begin{align}
\nabla J(W) &= \frac{1}{m}(h_W(X) - Y)X
\end{align}

Let's write the code for a single gradient descent step:
#
---

### **Log-odds:**

In the context of logistic regression, the log-odds is a **linear function** of the input features.

This is the fundamental reason why logistic regression is considered a **Generalized Linear Model (GLM)**.

#### The Mathematical Proof
In logistic regression, we model the probability $p$ using the sigmoid function:
$$p = \frac{1}{1 + e^{-z}}$$

If we solve this equation for $z$ (the logit function), we get:
$$z = \ln\left(\frac{p}{1-p}\right)$$

In a logistic regression model, $z$ is defined as the linear combination of inputs:
$$z = \beta_0 + \beta_1x_1 + \beta_2x_2 + \dots + \beta_nx_n$$

Because $z$ is a sum of features multiplied by weights, it is a **linear equation**.

#### Why this distinction matters:
1.  **Probability is Non-Linear:** If you try to model the probability $p$ directly as a linear function ($p = \beta x$), your model might predict values less than 0 or greater than 1, which is mathematically impossible for a probability.
2.  **Log-Odds is Linear:** By modeling the **log-odds** instead, the relationship between the features and the target is linear, while the resulting probability $p$ is constrained between 0 and 1 via the sigmoid transformation.

**Summary:**
*   **Relationship between features and probability ($p$):** Non-linear (S-shaped curve).
*   **Relationship between features and log-odds ($z$):** **Linear** (a straight line).
#
---

### **Process to update the W:**

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
# Here are our inputs.
X = np.array([[1, 3, -2, 0],
              [1, 1, 0, 1]])
Y = np.array([0, 1])

# initialize W
W = [1, 1, 1, 1]

In [4]:
# sigmpod function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# process to update the W
m, n = X.shape
learnign_rate = 0.1

# 1) Calculate the net_input
net_input = np.dot(X, W)

# 2) Sigmoid activation function
preds = sigmoid(net_input)

# 3) loss error
loss = -(Y * np.log(preds) + (1 - Y) * np.log(1 - preds)).mean()

# 4) gradient
gradient = np.dot((preds - Y), X) / m

# 5) update the weights
W = W - learnign_rate * gradient

In [6]:
print(f'predictions: {preds}')
print(f'loss: {loss}')
print(f'gradient: {gradient}')
print(f'weights: {W}')

predictions: [0.88079708 0.95257413]
loss: 1.0877576813083567
gradient: [ 0.4166856   1.29748268 -0.88079708 -0.02371294]
weights: [0.95833144 0.87025173 1.08807971 1.00237129]


### Tensorflow/keras

In [7]:
tf.keras.backend.clear_session # clear previous model from memory
model = tf.keras.Sequential()
model.add(
    tf.keras.layers.Dense(
        units=1,
        activation='sigmoid',
        use_bias=False,
        kernel_initializer=tf.ones_initializer
    )
)

optimizer = tf.keras.optimizers.SGD(learning_rate=0.1)
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
preds = model.predict(X)

history = model.fit(
    X = X,
    y = Y,
    epochs=1, # number of times the learning algorithm will work through the entire training dataset.
    batch_size=2,
    verbose=0
)

In [ ]:
loss = history.history['loss'][0] # [0] refers to the epochs 1

In [ ]:
weights = model.layers[0].get_weights()[0].T # [0] refers to the epochs 1